In [1]:
import pandas as pd
from datetime import datetime

In [2]:
# Read data
YouGov = pd.read_csv(
    "../data/australia.csv",
    na_values=[" ", "__NA__"],
    keep_default_na=True,
    low_memory=False
)

YouGov.shape

(53833, 513)

In [3]:
# Convert the endtime column to datetime format
YouGov["endtime"] = pd.to_datetime(YouGov["endtime"].str.split().str[0], format="%d/%m/%Y")

# Define the date range where consent for medical questions was not given
date_not_given = (YouGov["endtime"] >= "2021-02-19") & (YouGov["endtime"] <= "2021-10-18")

# Fill PHQ4 and d1_health variables with "NA"
for col in ["PHQ4_1", "PHQ4_2", "PHQ4_3", "PHQ4_4",
    "d1_health_1","d1_health_2","d1_health_3","d1_health_4","d1_health_5",
    "d1_health_6","d1_health_7","d1_health_8","d1_health_9","d1_health_10",
    "d1_health_11","d1_health_12","d1_health_13",
    "d1_health_98","d1_health_99"]:
    if col in YouGov.columns:
        YouGov.loc[date_not_given, col] = YouGov.loc[date_not_given, col].fillna("NA")

In [4]:
# Drop columns with missing values greater than the 10000
YouGov = YouGov.loc[:, YouGov.isna().sum() <= 11000]

YouGov.shape

(53833, 53)

In [5]:
# Remove all remaining rows with missing values
YouGov = YouGov.dropna()

YouGov.shape

(40872, 53)

In [6]:
YouGov.columns

Index(['RecordNo', 'endtime', 'qweek', 'i2_health', 'i9_health', 'i11_health',
       'i12_health_1', 'i12_health_2', 'i12_health_3', 'i12_health_4',
       'i12_health_5', 'i12_health_6', 'i12_health_7', 'i12_health_8',
       'i12_health_11', 'i12_health_12', 'i12_health_13', 'i12_health_14',
       'i12_health_15', 'i12_health_16', 'd1_health_1', 'd1_health_2',
       'd1_health_3', 'd1_health_4', 'd1_health_5', 'd1_health_6',
       'd1_health_7', 'd1_health_8', 'd1_health_9', 'd1_health_10',
       'd1_health_11', 'd1_health_12', 'd1_health_13', 'd1_health_98',
       'd1_health_99', 'weight', 'age', 'gender', 'state', 'household_size',
       'employment_status', 'WCRex2', 'cantril_ladder', 'PHQ4_1', 'PHQ4_2',
       'PHQ4_3', 'PHQ4_4', 'WCRex1', 'i12_health_22', 'i12_health_23',
       'i12_health_25', 'r1_1', 'r1_2'],
      dtype='object')

In [7]:
# Create 2-week survey period index
start_date = YouGov["endtime"].min()
YouGov["week_number"] = ((YouGov["endtime"] - start_date).dt.days // 14) + 1

In [8]:
# Convert household size to numeric values

for col in ["household_size"]:
    if col in YouGov.columns:
        YouGov[col] = YouGov[col].replace({
            "1": 1, "2": 2, "3": 3, "4": 4,
            "5": 5, "6": 6, "7": 7,
            "8 or more": 8,
            "Don't know": None,
            "Prefer not to say": None
        })

YouGov = YouGov.dropna(subset=["household_size"])
YouGov.shape

C:\Users\11849\AppData\Local\Temp\ipykernel_17412\4191059328.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  YouGov[col] = YouGov[col].replace({


(39890, 54)

In [9]:
# Convert agreement scale to numeric values
for col in ["r1_1", "r1_2"]:
    if col in YouGov.columns:
        YouGov[col] = YouGov[col].replace({
            "7 - Agree": 7,
            "6": 6,
            "5": 5,
            "4": 4,
            "3": 3,
            "2": 2,
            "1 – Disagree": 1
        })

C:\Users\11849\AppData\Local\Temp\ipykernel_17412\545370418.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  YouGov[col] = YouGov[col].replace({


In [10]:
# Map frequency responses to numeric scale
for col in YouGov.columns:
    if col.startswith("i12_health_"):
        YouGov[col] = YouGov[col].replace({
            "Always": 5,
            "Frequently": 4,
            "Sometimes": 3,
            "Rarely": 2,
            "Not at all": 1
        })

C:\Users\11849\AppData\Local\Temp\ipykernel_17412\1456475123.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  YouGov[col] = YouGov[col].replace({


In [11]:
mask_items = ["i12_health_1", "i12_health_22", "i12_health_23", "i12_health_25"]
YouGov["face_mask_behaviour_scale"] = YouGov[mask_items].median(axis=1)

# Convert the face mask behaviour scale into a binary variable
YouGov["face_mask_behaviour_binary"] = ""

for i in YouGov.index:
    value = YouGov.loc[i, "face_mask_behaviour_scale"]
    if value >= 4:
        YouGov.loc[i, "face_mask_behaviour_binary"] = 1
    else:
        YouGov.loc[i, "face_mask_behaviour_binary"] = 0

In [ ]:
# Collect all self-protective behaviour items
protective_behaviour_cols = []

for col in YouGov.columns:
    if col.startswith("i12_"):
        protective_behaviour_cols.append(col)

# Create the general protective behaviour scale from all self-protective behaviour items
YouGov["protective_behaviour_scale"] = YouGov[protective_behaviour_cols].median(axis=1)

# Convert the general protective behaviour scale into a binary variable
YouGov["protective_behaviour_binary"] = ""

for i in YouGov.index:
    value = YouGov.loc[i, "protective_behaviour_scale"]
    if value >= 4:
        YouGov.loc[i, "protective_behaviour_binary"] = 1
    else:
        YouGov.loc[i, "protective_behaviour_binary"] = 0

In [13]:
# Create the non-mask protective behaviour scale by excluding the four mask-related items
protective_behaviour_nomask_cols = []
for col in protective_behaviour_cols:
    if col not in mask_items:
        protective_behaviour_nomask_cols.append(col)

YouGov["protective_behaviour_nomask_scale"] = YouGov[protective_behaviour_nomask_cols].median(axis=1)

In [14]:
# Combine comorbidity variables into a single categorical variable
d1_health_cols = []
for col in YouGov.columns:
    if col.startswith("d1_health_"):
        d1_health_cols.append(col)


YouGov["d1_comorbidities"] = "Yes"

for idx in YouGov.index:
    if "d1_health_99" in YouGov.columns:
        if YouGov.loc[idx, "d1_health_99"] == "Yes":
            YouGov.loc[idx, "d1_comorbidities"] = "No"

    if "d1_health_98" in YouGov.columns:
        if YouGov.loc[idx, "d1_health_98"] == "Yes":
            YouGov.loc[idx, "d1_comorbidities"] = "Prefer not to say"

YouGov = YouGov.drop(d1_health_cols, axis=1)


In [15]:
print(YouGov.shape)
print(YouGov.columns)

(39890, 45)
Index(['RecordNo', 'endtime', 'qweek', 'i2_health', 'i9_health', 'i11_health',
       'i12_health_1', 'i12_health_2', 'i12_health_3', 'i12_health_4',
       'i12_health_5', 'i12_health_6', 'i12_health_7', 'i12_health_8',
       'i12_health_11', 'i12_health_12', 'i12_health_13', 'i12_health_14',
       'i12_health_15', 'i12_health_16', 'weight', 'age', 'gender', 'state',
       'household_size', 'employment_status', 'WCRex2', 'cantril_ladder',
       'PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4', 'WCRex1', 'i12_health_22',
       'i12_health_23', 'i12_health_25', 'r1_1', 'r1_2', 'week_number',
       'face_mask_behaviour_scale', 'face_mask_behaviour_binary',
       'protective_behaviour_scale', 'protective_behaviour_binary',
       'protective_behaviour_nomask_scale', 'd1_comorbidities'],
      dtype='object')


In [16]:
YouGov = YouGov.drop(columns = ["RecordNo", "qweek", "weight"])
YouGov = YouGov.drop(columns = protective_behaviour_cols)
YouGov = YouGov.drop(columns = ['face_mask_behaviour_scale','protective_behaviour_scale'])


print(YouGov.shape)
print(YouGov.columns)

(39890, 23)
Index(['endtime', 'i2_health', 'i9_health', 'i11_health', 'age', 'gender',
       'state', 'household_size', 'employment_status', 'WCRex2',
       'cantril_ladder', 'PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4', 'WCRex1',
       'r1_1', 'r1_2', 'week_number', 'face_mask_behaviour_binary',
       'protective_behaviour_binary', 'protective_behaviour_nomask_scale',
       'd1_comorbidities'],
      dtype='object')


In [17]:
# Convert categorical variables into dummy variables
for col in ["state","gender","i9_health","employment_status","i11_health","WCRex1","WCRex2","PHQ4_1","PHQ4_2","PHQ4_3","PHQ4_4","d1_comorbidities"]:
    if col in YouGov.columns:
        dummies = pd.get_dummies(YouGov[col], prefix=col, drop_first=True)
        YouGov = pd.concat([YouGov, dummies], axis=1)
        YouGov = YouGov.drop(columns=[col])

YouGov.head(1)

,endtime,i2_health,age,household_size,cantril_ladder,r1_1,r1_2,week_number,face_mask_behaviour_binary,protective_behaviour_binary,...,PHQ4_3_Not at all,PHQ4_3_Prefer not to say,PHQ4_3_Several days,PHQ4_4_NA,PHQ4_4_Nearly every day,PHQ4_4_Not at all,PHQ4_4_Prefer not to say,PHQ4_4_Several days,d1_comorbidities_Prefer not to say,d1_comorbidities_Yes
9023,2020-06-24,0.0,31,1.0,8.0,5,1,1,0,0,...,False,False,True,False,False,False,False,False,False,True


In [18]:
YouGov = YouGov.replace({True: 1, False: 0})
YouGov.to_csv("../data/cleaned_YouGov.csv", index=False)

C:\Users\11849\AppData\Local\Temp\ipykernel_17412\3112089125.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  YouGov = YouGov.replace({True: 1, False: 0})


In [19]:
print(YouGov.shape)
print(YouGov.columns)

(39890, 60)
Index(['endtime', 'i2_health', 'age', 'household_size', 'cantril_ladder',
       'r1_1', 'r1_2', 'week_number', 'face_mask_behaviour_binary',
       'protective_behaviour_binary', 'protective_behaviour_nomask_scale',
       'state_New South Wales', 'state_Northern Territory', 'state_Queensland',
       'state_South Australia', 'state_Tasmania', 'state_Victoria',
       'state_Western Australia', 'gender_Male', 'i9_health_Not sure',
       'i9_health_Yes', 'employment_status_Not working',
       'employment_status_Part time employment', 'employment_status_Retired',
       'employment_status_Unemployed', 'i11_health_Not sure',
       'i11_health_Somewhat unwilling', 'i11_health_Somewhat willing',
       'i11_health_Very unwilling', 'i11_health_Very willing',
       'WCRex1_Somewhat badly', 'WCRex1_Somewhat well', 'WCRex1_Very badly',
       'WCRex1_Very well', 'WCRex2_A lot of confidence', 'WCRex2_Don't know',
       'WCRex2_No confidence at all', 'WCRex2_Not very much confid

In [20]:
YouGov.head(1)

,endtime,i2_health,age,household_size,cantril_ladder,r1_1,r1_2,week_number,face_mask_behaviour_binary,protective_behaviour_binary,...,PHQ4_3_Not at all,PHQ4_3_Prefer not to say,PHQ4_3_Several days,PHQ4_4_NA,PHQ4_4_Nearly every day,PHQ4_4_Not at all,PHQ4_4_Prefer not to say,PHQ4_4_Several days,d1_comorbidities_Prefer not to say,d1_comorbidities_Yes
9023,2020-06-24,0.0,31,1.0,8.0,5,1,1,0,0,...,0,0,1,0,0,0,0,0,0,1
